# Train the GuardianEye image branch

Run this on a GPU runtime (Colab or Kaggle). The Space or the Docker host only serves weights; it does not train.

One YOLOv8 run is started per domain (`aerial_thermal` and `camera_trap_rgb`). Do not average those runs into one number. Metrics are written by `python -m guardianeye.eval.image_eval` into `reports/image/`. Copy them from those files. Do not type them into the notebook output by hand.

Fill dataset URLs in `configs/data.yaml` before the download cell. They ship as `TODO`.

In [ ]:
!git clone https://github.com/YOUR_USER/guardianeye.git
%cd guardianeye
!pip install -r requirements.txt && pip install -e .

In [ ]:
# Edit configs/data.yaml so downloads.*.url is a real link, then:
!python scripts/download_data.py
!python scripts/prepare_image_data.py

In [ ]:
# Drop --smoke for the full run defined in configs/image.yaml.
!python -m guardianeye.image.train
!python -m guardianeye.eval.image_eval

In [ ]:
import os
from huggingface_hub import HfApi

repo_id = os.environ["HF_MODEL_REPO"]  # e.g. your-user/guardianeye-weights
api = HfApi()
api.create_repo(repo_id, repo_type="model", exist_ok=True)
for name in ("aerial_thermal", "camera_trap_rgb"):
    api.upload_file(
        path_or_fileobj=f"models/image/{name}.pt",
        path_in_repo=f"image/{name}.pt",
        repo_id=repo_id,
        repo_type="model",
    )
    api.upload_file(
        path_or_fileobj=f"models/image/{name}.json",
        path_in_repo=f"image/{name}.json",
        repo_id=repo_id,
        repo_type="model",
    )
print("uploaded image checkpoints to", repo_id)